**Install Required Packages**

In [1]:
%pip install google-cloud-bigquery google-cloud-storage google-auth pandas db_dtypes


  Using cached google_cloud_bigquery-3.40.1-py3-none-any.whl.metadata (8.2 kB)
  Using cached google_api_core-2.30.0-py3-none-any.whl.metadata (3.1 kB)
  Using cached google_cloud_core-2.5.0-py3-none-any.whl.metadata (3.1 kB)
  Using cached google_resumable_media-2.8.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached google_crc32c-1.8.0-cp313-cp313-win_amd64.whl.metadata (1.8 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached cffi-2.0.0-cp313-cp313-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
Using cached google_cloud_bigquery-3.40.1-py3-none-any.whl (262 kB)
Using cached google_api_core-2.30.0-py3-none-any.whl (173 kB)
Using cached g


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


**Import libraries and initialize client instances**


In [1]:
import os
import asyncio
import google.auth

# os.environ['GOOGLE_CLOUD_PROJECT']

GOOGLE_CLOUD_PROJECT = "us-gcp-ame-con-e5556-npd-1"
GOOGLE_CLOUD_LOCATION = "us-central1"
MODEL_NAME = "gemini-2.5-flash"

os.environ['GOOGLE_CLOUD_PROJECT'] = GOOGLE_CLOUD_PROJECT
os.environ['GOOGLE_CLOUD_LOCATION'] = GOOGLE_CLOUD_LOCATION
os.environ['MODEL_NAME'] = MODEL_NAME

google.auth.default()

(<google.oauth2.credentials.Credentials at 0x2900ca9c980>,
 'us-gcp-ame-con-e5556-npd-1')

In [2]:
os.getenv("GOOGLE_CLOUD_PROJECT")
os.getenv("GOOGLE_CLOUD_LOCATION")

'us-central1'

In [3]:
from google.cloud import bigquery
from google.cloud import storage

# Initialize BigQuery and Storage clients
bigquery_client = bigquery.Client()
storage_client = storage.Client()

# --- Configuration for your data load ---
# Replace with your project ID, dataset ID, table ID, and GCS bucket details
project_id = "us-gcp-ame-con-e5556-npd-1"
location = "us-central1"
dataset_id = "emergency_response"
raw_table_id = "airports_data_raw"
gold_table_id = "airports_alerts_data"
model_id = "public_communication_model"
source_bucket_name = "gs://labs.roitraining.com/data-to-ai-workshop"
source_file_name = "gs://labs.roitraining.com/data-to-ai-workshop/airports.csv"

**Create Dataset**

In [4]:
sql_ddl_dataset_statement = f"""
CREATE SCHEMA IF NOT EXISTS `{project_id}.{dataset_id}`
OPTIONS(
    location = '{location}',
    default_table_expiration_days=14
);
"""

# Execute the DDL statement using the BigQuery client
query_job = bigquery_client.query(sql_ddl_dataset_statement)

# Wait for the job to complete
query_job.result()

print(f"Dataset '{dataset_id}' created or already exists in project '{project_id}' at location '{location}'.")


Dataset 'emergency_response' created or already exists in project 'us-gcp-ame-con-e5556-npd-1' at location 'us-central1'.


**Load data from Storage bucket into Big Query Table**

In [10]:
raw_table_ref = f"{project_id}.{dataset_id}.{raw_table_id}"

# Configure the load job
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,  # Assuming the CSV has a header row
    autodetect=True,      # Let BigQuery autodetect schema
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE, # Overwrite if table exists
)

# Start the load job
load_job = bigquery_client.load_table_from_uri(
    source_file_name, raw_table_ref, job_config=job_config
)

# Wait for the job to complete
load_job.result()

print(f"Data loaded from '{source_file_name}' into BigQuery table '{raw_table_ref}'.")

Data loaded from 'gs://labs.roitraining.com/data-to-ai-workshop/airports.csv' into BigQuery table 'us-gcp-ame-con-e5556-npd-1.emergency_response.airports_data_raw'.


In [11]:
import pandas
df = bigquery_client.list_rows(f"{raw_table_ref}").to_dataframe()
df


c:\Workspace\.venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11,NA,US,US-PA,Bensalem,False,None,None,K00A,00A,https://www.penndot.pa.gov/TravelInPA/airports...,None,None
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435,NA,US,US-KS,Leoti,False,None,None,00AA,00AA,None,None,None
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450,NA,US,US-AK,Anchor Point,False,None,None,00AK,00AK,None,None,None
3,6525,00AL,small_airport,Epps Airpark,34.864799,-86.770302,820,NA,US,US-AL,Harvest,False,None,None,00AL,00AL,None,None,None
4,506791,00AN,small_airport,Katmai Lodge Airport,59.093287,-156.456699,80,NA,US,US-AK,King Salmon,False,None,None,00AN,00AN,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82888,32753,ZYYY,medium_airport,Shenyang Dongta Airport,41.784401,123.496002,<NA>,AS,CN,CN-21,"Dadong, Shenyang",False,None,None,ZYYY,None,None,None,None
82889,46378,ZZ-0001,heliport,Sealand Helipad,51.894444,1.482500,40,EU,GB,GB-ENG,Sealand,False,None,None,None,None,http://www.sealandgov.org/,https://en.wikipedia.org/wiki/Principality_of_...,Roughs Tower Helipad
82890,307326,ZZ-0002,small_airport,Glorioso Islands Airstrip,-11.584278,47.296389,11,AF,TF,TF-U-A,Grande Glorieuse,False,None,None,None,None,None,None,None
82891,346788,ZZ-0003,small_airport,Fainting Goat Airport,32.110587,-97.356312,690,NA,US,US-TX,Blum,False,None,None,87TX,87TX,None,None,None


In [12]:
df.groupby('type')['name'].unique()


type
balloonport       [Clinton Elks Lodge Balloonport, Aeronut Park ...
closed            [Ring Hill Airport, Land's Field, Flying S Ran...
heliport          [Total RF Heliport, Kitchen Creek Helibase Hel...
large_airport     [Honiara International Airport, Port Moresby J...
medium_airport    [Aleknagik / New Airport, Sas Al Nakheel Air B...
seaplane_base     [Cliche Cove Seaplane Base, Annapolis Seaplane...
small_airport     [Aero B Ranch Airport, Lowell Field, Epps Airp...
Name: name, dtype: object

**Filter Large Airports in US**

In [13]:
large_airports_us = df[(df['type'] == 'large_airport') & (df['iso_country'] == 'US')]
large_airports_us


,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
37412,16091,KABQ,large_airport,Albuquerque International Sunport,35.039976,-106.608925,5355,NA,US,US-NM,Albuquerque,True,KABQ,ABQ,KABQ,ABQ,http://www.abqsunport.com/,https://en.wikipedia.org/wiki/Albuquerque_Inte...,None
37431,3364,KADW,large_airport,Joint Base Andrews,38.810799,-76.866997,280,NA,US,US-MD,Camp Springs,False,KADW,ADW,KADW,ADW,http://www.jba.af.mil/,https://en.wikipedia.org/wiki/Joint_Base_Andrews,Andrews Air Force Base
37532,3384,KATL,large_airport,Hartsfield Jackson Atlanta International Airport,33.636700,-84.428101,1026,NA,US,US-GA,Atlanta,True,KATL,ATL,KATL,ATL,http://www.atlanta-airport.com/,https://en.wikipedia.org/wiki/Hartsfield–Jacks...,None
37541,3386,KAUS,large_airport,Austin Bergstrom International Airport,30.197535,-97.662015,542,NA,US,US-TX,Austin,True,KAUS,AUS,KAUS,AUS,http://www.ci.austin.tx.us/austinairport/,https://en.wikipedia.org/wiki/Austin-Bergstrom...,"BSM, KBSM, Bergstrom AFB"
37588,3396,KBDL,large_airport,Bradley International Airport,41.938510,-72.688066,173,NA,US,US-CT,Hartford,True,KBDL,BDL,KBDL,BDL,http://www.bradleyairport.com/,https://en.wikipedia.org/wiki/Bradley_Internat...,"HFD, Hartford"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41558,3926,KTPA,large_airport,Tampa International Airport,27.975500,-82.533203,26,NA,US,US-FL,Tampa,True,KTPA,TPA,KTPA,TPA,https://www.tampaairport.com/,https://en.wikipedia.org/wiki/Tampa_Internatio...,None
41577,3930,KTUL,large_airport,Tulsa International Airport,36.198399,-95.888100,677,NA,US,US-OK,Tulsa,True,KTUL,TUL,KTUL,TUL,None,https://en.wikipedia.org/wiki/Tulsa_Internatio...,None
51309,5388,PANC,large_airport,Ted Stevens Anchorage International Airport,61.179004,-149.992561,152,NA,US,US-AK,Anchorage,True,PANC,ANC,PANC,ANC,https://dot.alaska.gov/anc/,https://en.wikipedia.org/wiki/Ted_Stevens_Anch...,None
52380,5453,PHNL,large_airport,Daniel K Inouye International Airport,21.320620,-157.924228,13,NA,US,US-HI,"Honolulu, Oahu",True,PHNL,HNL,PHNL,HNL,http://airports.hawaii.gov/hnl/,https://en.wikipedia.org/wiki/Daniel_K._Inouye...,"Hickam Air Force Base, HIK, PHIK, KHNL, Honolu..."


**Retrieve Forecast URLs for each airport and append it to the dataframe**

In [14]:
import requests
headers={'userAgent':'us-airport-weather-forecast', 'accept': 'application/geo+json'}
def get_forecast_url(latitude, longitude):
    """
    Retrieves the forecast URL for a given latitude and longitude using the
    api.weather.gov points endpoint.

    Args:
        latitude (float): The latitude of the location.
        longitude (float): The longitude of the location.

    Returns:
        str: The forecast URL, or None if an error occurs.
    """

    url = f"https://api.weather.gov/points/{latitude},{longitude}"
    try:
        response = requests.get(url,headers=headers)
        response.raise_for_status()  # Raise an exception for bad status codes
        data = response.json()
        forecast_url = data.get("properties", {}).get("forecast")
        print(f"Forecast URL for {latitude},{longitude}: {forecast_url}")
        if forecast_url:
            return forecast_url
        else:
            return ""
    except requests.exceptions.RequestException as e:
        print(f"Error retrieving forecast URL for {latitude},{longitude}: {e}")
        return None

# Apply the function to the large_airports_us dataframe
large_airports_us['forecast_url'] = large_airports_us.apply(
    lambda row: get_forecast_url(row['latitude_deg'], row['longitude_deg']),
    axis=1
)
# forecast_url = get_forecast_url(40.070985, -74.933689)
# print(forecast_url)
print("Forecast URLs appended to the dataframe.")
print(large_airports_us[['name', 'forecast_url']].head())


Forecast URL for 35.039976,-106.608925: https://api.weather.gov/gridpoints/ABQ/100,119/forecast
Forecast URL for 38.810799,-76.866997: https://api.weather.gov/gridpoints/LWX/103,68/forecast
Forecast URL for 33.6367,-84.428101: https://api.weather.gov/gridpoints/FFC/50,82/forecast
Forecast URL for 30.197535,-97.662015: https://api.weather.gov/gridpoints/EWX/159,88/forecast
Forecast URL for 41.93851,-72.688066: https://api.weather.gov/gridpoints/BOX/20,61/forecast
Forecast URL for 36.1245,-86.6782: https://api.weather.gov/gridpoints/OHX/54,55/forecast
Forecast URL for 42.36197,-71.0079: https://api.weather.gov/gridpoints/BOX/73,90/forecast
Forecast URL for 42.940498,-78.732201: https://api.weather.gov/gridpoints/BUF/40,50/forecast
Forecast URL for 39.1754,-76.668297: https://api.weather.gov/gridpoints/LWX/108,86/forecast
Forecast URL for 41.411701,-81.8498: https://api.weather.gov/gridpoints/CLE/78,61/forecast
Forecast URL for 35.2140007019043,-80.94309997558594: https://api.weather.gov/

C:\Users\kmurugappan\AppData\Local\Temp\ipykernel_33516\3956242721.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  large_airports_us['forecast_url'] = large_airports_us.apply(


In [16]:
import pandas as pd

def get_forecast_data(forecast_url):
    """
    Fetches forecast data from a given URL.

    Args:
        forecast_url (str): The URL to fetch forecast data from.

    Returns:
        dict: The JSON response from the URL, or None if an error occurs.
    """
    if not forecast_url:
        return None
    try:
        response = requests.get(forecast_url, headers=headers)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching forecast data from {forecast_url}: {e}")
        return None

def extract_first_three_periods(forecast_data):
    """
    Extracts the first three periods of forecast data.

    Args:
        forecast_data (dict): The JSON response containing forecast data.

    Returns:
        list: A list of dictionaries, each representing a forecast period.
    """
    if not forecast_data or 'properties' not in forecast_data or 'periods' not in forecast_data['properties']:
        return []
    return forecast_data['properties']['periods'][:3]

def prepare_data_for_bq(airport_data, forecast_periods):
    """
    Prepares data for BigQuery insertion.

    Args:
        airport_data (pd.Series): A Pandas Series containing airport information.
        forecast_periods (list): A list of forecast period dictionaries.

    Returns:
        list: A list of dictionaries, each representing a row for BigQuery.
    """
    rows_to_insert = []
    for period in forecast_periods:
        row = {
            "airport_name": airport_data['name'],
            "airport_iata_code": airport_data['iata_code'],
            "airport_latitude": airport_data['latitude_deg'],
            "airport_longitude": airport_data['longitude_deg'],
            "forecast_period_name": period.get('name'),
            "forecast_period_start_time": pd.to_datetime(period.get('startTime'), errors='coerce'),
            "forecast_period_end_time": pd.to_datetime(period.get('endTime'), errors='coerce'),
            "forecast_period_temperature": period.get('temperature'),
            "forecast_period_temperature_unit": period.get('temperatureUnit'),
            "forecast_period_wind_speed": period.get('windSpeed'),
            "forecast_period_wind_direction": period.get('windDirection'),
            "forecast_period_short_forecast": period.get('shortForecast'),
            "forecast_period_detailed_forecast": period.get('detailedForecast'),
            "forecast_period_icon": period.get('icon')
        }
        rows_to_insert.append(row)
    return rows_to_insert

# Initialize an empty list to store all rows for BigQuery
all_rows_for_bq = []

# Iterate through each airport in the filtered dataframe
for index, row in large_airports_us.iterrows():
    forecast_data = get_forecast_data(row['forecast_url'])
    if forecast_data:
        first_three_periods = extract_first_three_periods(forecast_data)
        prepared_rows = prepare_data_for_bq(row, first_three_periods)
        all_rows_for_bq.extend(prepared_rows)

# Convert the list of dictionaries to a Pandas DataFrame
gold_df = pd.DataFrame(all_rows_for_bq)

# Define the BigQuery table reference for the gold table
gold_table_ref = f"{project_id}.{dataset_id}.{gold_table_id}"

# Configure the load job for the gold table
gold_job_config = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("airport_name", "STRING"),
        bigquery.SchemaField("airport_iata_code", "STRING"),
        bigquery.SchemaField("airport_latitude", "FLOAT"),
        bigquery.SchemaField("airport_longitude", "FLOAT"),
        bigquery.SchemaField("forecast_period_name", "STRING"),
        bigquery.SchemaField("forecast_period_start_time", "TIMESTAMP"),
        bigquery.SchemaField("forecast_period_end_time", "TIMESTAMP"),
        bigquery.SchemaField("forecast_period_temperature", "INTEGER"),
        bigquery.SchemaField("forecast_period_temperature_unit", "STRING"),
        bigquery.SchemaField("forecast_period_wind_speed", "STRING"),
        bigquery.SchemaField("forecast_period_wind_direction", "STRING"),
        bigquery.SchemaField("forecast_period_short_forecast", "STRING"),
        bigquery.SchemaField("forecast_period_detailed_forecast", "STRING"),
        bigquery.SchemaField("forecast_period_icon", "STRING")
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)


# Start the load job for the gold table
gold_load_job = bigquery_client.load_table_from_dataframe(
    gold_df, gold_table_ref, job_config=gold_job_config
)

# Wait for the job to complete
gold_load_job.result()

print(f"Data loaded into BigQuery table '{gold_table_ref}'.")

c:\Workspace\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Data loaded into BigQuery table 'us-gcp-ame-con-e5556-npd-1.emergency_response.airports_alerts_data'.


**Create ML Model using Gemini LLM**

In [17]:
create_remote_model_statement = f"""
CREATE OR REPLACE MODEL `{project_id}.{dataset_id}.{model_id}`
REMOTE WITH CONNECTION `{location}.gemini_connection`
OPTIONS (
  ENDPOINT = 'gemini-2.5-flash'
);
"""

# Execute the DDL statement using the BigQuery client
query_job = bigquery_client.query(create_remote_model_statement)

# Wait for the job to complete
query_job.result()

print(f"Remote model '{model_id}' created or replaced in dataset '{dataset_id}'.")

Remote model 'public_communication_model' created or replaced in dataset 'emergency_response'.


**Create Weather Report for each Airport in US**

In [20]:
sql_query_statement = f"""
SELECT ml_generate_text_llm_result as weather_alert,
  * Except(ml_generate_text_status, ml_generate_text_llm_result, ml_generate_text_rai_result)
FROM
  ML.GENERATE_TEXT(MODEL `{project_id}.{dataset_id}.{model_id}`,
    (
      SELECT
        *,
        Concat('Generate a concise weather alert for the airport in JSON format with the following keys: "alert_type" (e.g., "Warning", "Advisory", "Information"), "summary" (a brief description of the weather event), "visibility" (VFR - Visual Flight Rules: > 5 SM, MVFR - Marginal VFR: 3 to 5 SM,IFR - Instrument Flight Rules: 1 to < 3 SM, LIFR -Low IFR: < 1 SM), "severity" (Light, Moderate, Heavy, Extreme), "runway_condition" (Slush, Ice, Snow, Dry, Wet) and "recommendation" (actions to take or advice). Use the following data: Airport: ' , airport_name, ' Period: ', forecast_period_name, 'Period Start Time:', forecast_period_start_time, 'Period End Time:', forecast_period_end_time, 'Temperature: ', forecast_period_temperature, 'Wind Speed: ', forecast_period_wind_speed,'Wind Direction: ', forecast_period_wind_direction, 'Detailed Forecast: ', forecast_period_detailed_forecast ) AS prompt
      FROM
        `{project_id}.{dataset_id}.{gold_table_id}`
    ),
    STRUCT(
      0.2 AS temperature,
      0.9 AS top_p,
      1000 AS max_output_tokens,
      TRUE AS flatten_json_output
    )
  );
"""

# Execute the query and return a DataFrame
df_alerts = bigquery_client.query(sql_query_statement).to_dataframe()

import json

def parse_weather_report(report):
    try:
        # Some LLMs might wrap JSON in markdown blocks
        clean_report = report.replace('```json', '').replace('```', '').strip()
        data = json.loads(clean_report)
        return pd.Series(data)
    except:
        return pd.Series({
            "alert_type": None, 
            "summary": None, 
            "visibility": None, 
            "severity": None, 
            "runway_condition": None, 
            "recommendation": None
        })

# Extract JSON fields into individual columns
df_insights = pd.concat([df_alerts, df_alerts['weather_alert'].apply(parse_weather_report)], axis=1)

# Drop the raw weather_alert and prompt columns if they exist
df_insights = df_insights.drop(columns=['weather_alert', 'prompt'], errors='ignore')

# Store the final data frame into a new table
insights_table_id = "airport_alerts_insights"
insights_table_ref = f"{project_id}.{dataset_id}.{insights_table_id}"

job_config = bigquery.LoadJobConfig(write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE)
bigquery_client.load_table_from_dataframe(df_insights, insights_table_ref, job_config=job_config).result()

print(f"Weather insights stored in table '{insights_table_id}'.")
df_insights.head()

c:\Workspace\.venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
c:\Workspace\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Weather insights stored in table 'airport_alerts_insights'.


,weather_report,airport_name,airport_iata_code,airport_latitude,airport_longitude,forecast_period_name,forecast_period_start_time,forecast_period_end_time,forecast_period_temperature,forecast_period_temperature_unit,...,forecast_period_icon,alert_type,summary,visibility,severity,runway_condition,recommendation,airport,period_start_time,period_end_time
0,**Weather Report for Daniel K. Inouye Internat...,Daniel K Inouye International Airport,HNL,21.320620,-157.924228,Tonight,2026-03-29 04:00:00+00:00,2026-03-29 16:00:00+00:00,69,F,...,https://api.weather.gov/icons/land/night/wind_...,Advisory,Gusty winds expected tonight at Daniel K. Inou...,VFR,Moderate,Dry,Pilots should be aware of potential crosswinds...,NaN,NaN,NaN
1,**Weather Report: Daniel K. Inouye Internation...,Daniel K Inouye International Airport,HNL,21.320620,-157.924228,Today,2026-03-28 17:00:00+00:00,2026-03-29 04:00:00+00:00,75,F,...,https://api.weather.gov/icons/land/day/wind_sc...,Information,Gusty winds expected at Daniel K. Inouye Inter...,VFR,Moderate,Dry,Pilots should be aware of potential wind shear...,NaN,NaN,NaN
2,**Weather Report for Daniel K. Inouye Internat...,Daniel K Inouye International Airport,HNL,21.320620,-157.924228,Sunday,2026-03-29 16:00:00+00:00,2026-03-30 04:00:00+00:00,75,F,...,https://api.weather.gov/icons/land/day/wind_fe...,Information,Gusty winds expected at Daniel K. Inouye Inter...,VFR,Moderate,Dry,Pilots should be aware of potential crosswinds...,NaN,NaN,NaN
3,**Weather Report: Kahului International Airpor...,Kahului International Airport,OGG,20.896263,-156.431837,Today,2026-03-28 17:00:00+00:00,2026-03-29 04:00:00+00:00,76,F,...,https://api.weather.gov/icons/land/day/rain_sh...,Advisory,Rain showers and gusty winds expected at Kahul...,MVFR,Moderate,Wet,Pilots should be aware of reduced visibility a...,NaN,NaN,NaN
4,**Weather Report for Kahului International Air...,Kahului International Airport,OGG,20.896263,-156.431837,Sunday,2026-03-29 16:00:00+00:00,2026-03-30 04:00:00+00:00,76,F,...,https://api.weather.gov/icons/land/day/sct?siz...,Information,Gusty winds expected at Kahului International ...,VFR,Moderate,Dry,Pilots should be aware of potential crosswinds...,NaN,NaN,NaN
